<a href="https://colab.research.google.com/github/taqiyyaindartono/Final_Project_ir/blob/main/01-corpus.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!git config --global user.email "fauzan1464@gmail.com"
!git config --global user.name "fauzanrahmatullah"

In [4]:
import pandas as pd

url = "https://raw.githubusercontent.com/taqiyyaindartono/Final_Project_ir/refs/heads/main/jobs_clean.csv"
df = pd.read_csv(url, sep=",")

# print jumlah baris, kolom, dan 5 data teratas
print(f"Total dokumen: {df.shape[0]} baris, {df.shape[1]} kolom")
df.head()

Total dokumen: 3600 baris, 21 kolom


,job_id,snapshot_date,job_title,role_family,seniority,company_size,industry,city,province,work_mode,...,salary_min_idr,salary_max_idr,salary_mid_idr,currency,skills,tools,experience_years_min,education_level,posting_age_days,is_remote_friendly
0,IDJOB202602205,2026-01-01,Reporting Analyst,BI Analyst,Junior,1000+,Fintech,Bandung,West Java,Hybrid,...,6500000,12500000,9500000,IDR,Dashboarding; Data Modeling; ETL; Git; Problem...,Power BI; Slack; Tableau,1,Bachelor,51,True
1,IDJOB202600149,2026-01-01,Senior Reporting Analyst,BI Analyst,Senior,201-500,HealthTech,Jakarta,DKI Jakarta,Hybrid,...,17500000,28500000,23000000,IDR,Dashboarding; Data Literacy; Data Modeling; ET...,BigQuery; Power BI; Slack,5,Bachelor,47,True
2,IDJOB202603289,2026-01-01,BI Analyst,BI Analyst,Junior,501-1000,Banking,Jakarta,DKI Jakarta,Hybrid,...,8000000,15000000,11500000,IDR,Data Modeling; ETL; SQL; Stakeholder Management,Tableau; dbt,1,Diploma,31,True
3,IDJOB202601308,2026-01-01,BI Analyst,BI Analyst,Mid,501-1000,Banking,Remote - Indonesia,Remote,Remote,...,9000000,16000000,12500000,IDR,Dashboarding; Git; Problem Solving; Stakeholde...,Power BI; dbt,3,Bachelor,43,True
4,IDJOB202601049,2026-01-01,Senior Business Intelligence Analyst,BI Analyst,Senior,501-1000,Travel,Tangerang,Banten,Hybrid,...,14500000,24500000,19500000,IDR,Communication; Dashboarding; Data Literacy; Da...,GitHub; Notion; Power BI; dbt,5,Bachelor,17,True


In [5]:
# cek nama kolom dan jumlah nilai kosong
print("Daftar Kolom:")
print(df.columns.tolist())

print("\nJumlah Nilai Kosong per Kolom:")
print(df.isnull().sum())

Daftar Kolom:
['job_id', 'snapshot_date', 'job_title', 'role_family', 'seniority', 'company_size', 'industry', 'city', 'province', 'work_mode', 'employment_type', 'salary_min_idr', 'salary_max_idr', 'salary_mid_idr', 'currency', 'skills', 'tools', 'experience_years_min', 'education_level', 'posting_age_days', 'is_remote_friendly']

Jumlah Nilai Kosong per Kolom:
job_id                  0
snapshot_date           0
job_title               0
role_family             0
seniority               0
company_size            0
industry                0
city                    0
province                0
work_mode               0
employment_type         0
salary_min_idr          0
salary_max_idr          0
salary_mid_idr          0
currency                0
skills                  0
tools                   0
experience_years_min    0
education_level         0
posting_age_days        0
is_remote_friendly      0
dtype: int64


In [6]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')

# 1. kolom isi lowongan
text_cols = [
    'job_title', 'role_family', 'seniority', 'industry',
    'city', 'province', 'work_mode', 'skills', 'tools'
]

# isi nilai kosong dg string kosong agar tidak eror
df_corpus = df.copy()
for col in text_cols:
    if col in df_corpus.columns:
        df_corpus[col] = df_corpus[col].fillna('').astype(str)

# gabung dokumen, bersihkan ;
df_corpus['document_text'] = df_corpus[text_cols].agg(' '.join, axis=1)
df_corpus['document_text'] = df_corpus['document_text'].str.replace(';', ' ', regex=False)
df_corpus['document_text'] = df_corpus['document_text'].str.replace(r'\s+', ' ', regex=True).str.strip()

# praproses baseline BM25
stop_words = set(stopwords.words('english'))

def preprocess_text(text):
    text = text.lower()                                    # case folding
    text = re.sub(r'[^a-z0-9\s\+\#]', ' ', text)           # keep simbol untuk bagian skill c++/c#
    tokens = word_tokenize(text)                           # pembuatan token
    tokens = [t for t in tokens if t not in stop_words and len(t) > 1]
    return tokens

df_corpus['tokens'] = df_corpus['document_text'].apply(preprocess_text)
df_corpus['token_count'] = df_corpus['tokens'].apply(len)

df_corpus[['job_id', 'job_title', 'document_text', 'token_count']].head()

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


,job_id,job_title,document_text,token_count
0,IDJOB202602205,Reporting Analyst,Reporting Analyst BI Analyst Junior Fintech Ba...,22
1,IDJOB202600149,Senior Reporting Analyst,Senior Reporting Analyst BI Analyst Senior Hea...,25
2,IDJOB202603289,BI Analyst,BI Analyst BI Analyst Junior Banking Jakarta D...,18
3,IDJOB202601308,BI Analyst,BI Analyst BI Analyst Mid Banking Remote - Ind...,19
4,IDJOB202601049,Senior Business Intelligence Analyst,Senior Business Intelligence Analyst BI Analys...,24


In [8]:
from collections import Counter

print("Statistik Dokumen:")
# panjang token
print(df_corpus['token_count'].describe())

# 15 term/kata paling sering muncul
all_tokens = [token for doc_tokens in df_corpus['tokens'] for token in doc_tokens]
top_terms = Counter(all_tokens).most_common(15)

print("\n15 Term Paling Sering Muncul")
for term, freq in top_terms:
    print(f"{term:<20}: {freq}")

Statistik Dokumen:
count    3600.000000
mean       21.793611
std         2.224523
min        15.000000
25%        20.000000
50%        22.000000
75%        23.000000
max        29.000000
Name: token_count, dtype: float64

15 Term Paling Sering Muncul
engineer            : 4037
data                : 3757
analyst             : 2208
jakarta             : 2182
git                 : 2104
problem             : 2097
solving             : 2097
communication       : 2070
literacy            : 2064
senior              : 1794
slack               : 1524
github              : 1510
mid                 : 1487
remote              : 1477
notion              : 1440


In [10]:
# simpan token versi siap, ditambah spasi biar gampang dibaca.
df_corpus['clean_text'] = df_corpus['tokens'].apply(lambda x: ' '.join(x))

output_cols = ['job_id', 'job_title', 'city', 'work_mode', 'salary_mid_idr', 'document_text', 'clean_text', 'token_count']
output_cols = [c for c in output_cols if c in df_corpus.columns]

df_corpus[output_cols].to_csv('corpus_ready.csv', index=False)
print("Berhasil menyimpan corpus_ready.csv!")

Berhasil menyimpan corpus_ready.csv!
